# 🌍 XRayEarth — xBD Dataset Exploration
**Seeing through disaster with satellite vision**

This notebook explores the xBD dataset:
- Class distribution (imbalance visualization)
- Sample pre/post image pairs
- Annotation statistics
- Tile generation preview

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from dotenv import load_dotenv

# Add src to path
sys.path.insert(0, str(Path('..') / 'src'))
load_dotenv('../.env')

from dataset import parse_xbd_annotation, DAMAGE_LABEL_MAP, CLASS_NAMES
from tiling  import generate_tile_coords, image_to_tiles

# Paths
DATA_DIR   = os.getenv('DATA_DIR', '../data')
PRE_DIR    = os.getenv('PRE_DIR',  '../data/pre')
POST_DIR   = os.getenv('POST_DIR', '../data/post')
LABELS_DIR = os.getenv('LABELS_DIR', '../data/labels')

print(f'Data dir: {DATA_DIR}')
print(f'Pre images:  {len(list(Path(PRE_DIR).glob("*.png")))}')
print(f'Post images: {len(list(Path(POST_DIR).glob("*.png")))}')
print(f'Labels:      {len(list(Path(LABELS_DIR).glob("*.json")))}')

## 1. Class Distribution — The Imbalance Problem

In [ ]:
# Count all building labels across dataset
label_counts = {name: 0 for name in CLASS_NAMES.values()}

for label_file in Path(LABELS_DIR).glob('*.json'):
    with open(label_file) as f:
        data = json.load(f)
    for feat in data.get('features', {}).get('xy', []):
        props = feat.get('properties', {})
        if props.get('feature_type') == 'building':
            subtype = props.get('subtype', 'no-damage').lower()
            cls_id  = DAMAGE_LABEL_MAP.get(subtype, 0)
            label_counts[CLASS_NAMES[cls_id]] += 1

print('Class distribution:')
total = sum(label_counts.values())
for cls, count in label_counts.items():
    pct = count / total * 100
    bar = '█' * int(pct / 2)
    print(f'  {cls:<15} {count:>6,}  ({pct:5.1f}%) {bar}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71', '#f39c12', '#e74c3c', '#8e44ad']
classes = list(label_counts.keys())
counts  = list(label_counts.values())

# Bar chart
axes[0].bar(classes, counts, color=colors)
axes[0].set_title('Class Distribution (absolute)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of buildings')
axes[0].tick_params(axis='x', rotation=15)
for i, (cls, cnt) in enumerate(zip(classes, counts)):
    axes[0].text(i, cnt + total*0.005, f'{cnt:,}', ha='center', fontsize=9)

# Pie chart
axes[1].pie(counts, labels=classes, colors=colors, autopct='%1.1f%%',
            startangle=90, pctdistance=0.85)
axes[1].set_title('Class Distribution (%)', fontsize=13, fontweight='bold')

plt.suptitle('xBD Dataset — Class Imbalance', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n⚠️  This extreme imbalance is WHY we need Focal Loss!')

## 2. Sample Pre/Post Image Pairs

In [ ]:
import random

# Get all image IDs
image_ids = [
    f.stem.replace('_pre_disaster', '')
    for f in Path(PRE_DIR).glob('*_pre_disaster.png')
]
random.seed(42)
sample_ids = random.sample(image_ids, min(3, len(image_ids)))

fig, axes = plt.subplots(len(sample_ids), 2, figsize=(14, 5 * len(sample_ids)))
if len(sample_ids) == 1:
    axes = [axes]

for row, img_id in enumerate(sample_ids):
    pre_img  = np.array(Image.open(Path(PRE_DIR)  / f'{img_id}_pre_disaster.png'))
    post_img = np.array(Image.open(Path(POST_DIR) / f'{img_id}_post_disaster.png'))

    # Load annotations
    label_path = Path(LABELS_DIR) / f'{img_id}_post_disaster.json'
    ann = parse_xbd_annotation(str(label_path), pre_img.shape[0], pre_img.shape[1])

    # Draw boxes
    colors_map = {1: 'green', 2: 'orange', 3: 'red', 4: 'purple'}

    for ax, img, title in zip(
        axes[row], [pre_img, post_img],
        [f'{img_id} — PRE', f'{img_id} — POST']
    ):
        ax.imshow(img)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.axis('off')

        for box, label in zip(ann['boxes'], ann['labels']):
            x1, y1, x2, y2 = box
            color = colors_map.get(label, 'white')
            rect  = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=1.5, edgecolor=color, facecolor='none'
            )
            ax.add_patch(rect)

plt.suptitle('Pre/Post Disaster Image Pairs with Annotations', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/sample_pairs.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Legend: Green=no-damage, Orange=minor, Red=major, Purple=destroyed')

## 3. Tiling Preview

In [ ]:
# Show how a full image gets tiled
sample_id = image_ids[0]
pre_img   = np.array(Image.open(Path(PRE_DIR) / f'{sample_id}_pre_disaster.png'))
H, W      = pre_img.shape[:2]

TILE_SIZE = 384
OVERLAP   = 0.15

coords = generate_tile_coords(W, H, TILE_SIZE, OVERLAP)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Full image with tile grid
axes[0].imshow(pre_img)
axes[0].set_title(f'Full Image ({W}×{H}) with {len(coords)} tiles', fontweight='bold')
for i, (x1, y1, x2, y2) in enumerate(coords):
    rect = patches.Rectangle(
        (x1, y1), x2-x1, y2-y1,
        linewidth=1, edgecolor='cyan', facecolor='cyan', alpha=0.1
    )
    axes[0].add_patch(rect)
    axes[0].text(x1+5, y1+20, str(i), color='cyan', fontsize=6)
axes[0].axis('off')

# First 6 tiles
axes[1].set_title(f'Tile Grid ({TILE_SIZE}×{TILE_SIZE}, {OVERLAP*100:.0f}% overlap)', 
                   fontweight='bold')
axes[1].imshow(pre_img)
colors_tile = plt.cm.Set3(np.linspace(0, 1, min(6, len(coords))))
for i, ((x1, y1, x2, y2), color) in enumerate(zip(coords[:6], colors_tile)):
    rect = patches.Rectangle(
        (x1, y1), x2-x1, y2-y1,
        linewidth=2, edgecolor=color, facecolor=color, alpha=0.3,
        label=f'Tile {i}'
    )
    axes[1].add_patch(rect)
axes[1].legend(loc='upper right', fontsize=8)
axes[1].axis('off')

plt.suptitle('Tiling Strategy — Preserving Full Resolution', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/tiling_preview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ {len(coords)} tiles generated from {W}×{H} image')
print(f'✓ Each tile: {TILE_SIZE}×{TILE_SIZE} with {OVERLAP*100:.0f}% overlap')

## 4. Focal Loss vs CrossEntropy Visualization

In [ ]:
import torch
sys.path.insert(0, str(Path('..') / 'src'))
from loss import FocalLoss, WeightedCrossEntropyLoss

# Visualize how focal weight changes with confidence
pt_values = np.linspace(0.01, 0.99, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Focal weight for different gamma values
for gamma in [0.5, 1.0, 2.0, 5.0]:
    focal_weights = (1 - pt_values) ** gamma
    axes[0].plot(pt_values, focal_weights, label=f'γ={gamma}', linewidth=2)

axes[0].axvline(x=0.8, color='gray', linestyle='--', alpha=0.7, label='High confidence (0.8)')
axes[0].set_xlabel('Model confidence (pt)', fontsize=12)
axes[0].set_ylabel('Focal weight (1-pt)^γ', fontsize=12)
axes[0].set_title('Focal Weight vs Confidence\n(γ=2.0 recommended)', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# CE vs Focal loss on imbalanced batch
np.random.seed(42)
n_easy, n_hard = 80, 20
easy_logits = torch.zeros(n_easy, 5); easy_logits[:, 1] = 4.0  # confident no-damage
hard_logits = torch.randn(n_hard, 5)                            # uncertain destroyed
easy_labels = torch.ones(n_easy, dtype=torch.long)
hard_labels = torch.full((n_hard,), 4, dtype=torch.long)

all_logits = torch.cat([easy_logits, hard_logits])
all_labels = torch.cat([easy_labels, hard_labels])

ce    = WeightedCrossEntropyLoss(reduction='none')
focal = FocalLoss(gamma=2.0, reduction='none')

ce_losses    = ce(all_logits, all_labels).detach().numpy()
focal_losses = focal(all_logits, all_labels).detach().numpy()

x = np.arange(len(all_labels))
axes[1].bar(x[:n_easy], ce_losses[:n_easy],    color='#3498db', alpha=0.7, label='CE — easy (no-damage)')
axes[1].bar(x[n_easy:], ce_losses[n_easy:],    color='#e74c3c', alpha=0.7, label='CE — hard (destroyed)')
axes[1].bar(x[:n_easy], focal_losses[:n_easy], color='#2ecc71', alpha=0.7, label='Focal — easy')
axes[1].bar(x[n_easy:], focal_losses[n_easy:], color='#e67e22', alpha=0.7, label='Focal — hard')
axes[1].set_title('CE vs Focal Loss per sample\n(Focal ignores easy, focuses on hard)', fontweight='bold')
axes[1].set_xlabel('Sample index')
axes[1].set_ylabel('Loss value')
axes[1].legend(fontsize=8)
axes[1].axvline(x=n_easy, color='black', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Why Focal Loss? — Handling Class Imbalance', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/focal_vs_ce.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key insight: Focal Loss near-zeroes the easy majority class loss!')

## 5. Dataset Statistics Summary

In [ ]:
print('=' * 50)
print('📊 xBD Dataset Summary')
print('=' * 50)
print(f'Total images    : {len(image_ids)}')
print(f'Total buildings : {sum(label_counts.values()):,}')
print()
print('Class breakdown:')
total = sum(label_counts.values())
for cls, count in label_counts.items():
    ratio = count / total
    focal_weight = (1 - ratio) ** 2
    print(f'  {cls:<15}: {count:>6,} ({ratio*100:5.1f}%) → focal_weight≈{focal_weight:.3f}')
print()
print('Imbalance ratio (no-damage / destroyed):', 
      f"{label_counts.get('no-damage',1) / max(label_counts.get('destroyed',1), 1):.1f}×")
print()
print('💡 This is why macro F1 is our primary metric!')
print('   Accuracy would be misleading with this imbalance.')